# Native engine comparisons
Replay **23.8 GHz ideal controls** and retain the difficult geometry diagnostics.
These historical controls used DiffeRT 0.12.0 and the recorded Sionna workstation
stack. They are not measurements of a newly installed bridge/runtime.
The exact input table and its original hash are bundled with this notebook.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks/replay.py").is_file())
sys.path.insert(0, str(ROOT / "notebooks"))
from replay import verify_data, read_json, dataframe
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.2, "figure.dpi": 110})
manifest = verify_data()
print(f"Verified {len(manifest['files'])} bundled historical data files.")
from visuals import show_image, compare_renders, verify_assets, ASSETS
verify_assets()


## 23.8 GHz ideal control geometry
This is the **two_leaves** control: two 0.8 × 0.8 m screens at x = −0.1/+0.1 m,
rotated −30°/+45°, with a 0.3 mm radio slab thickness. Red = TX, blue = RX.
The meshes are zero-thickness screens; the assigned radio slab thickness is
a material parameter. The other synthetic controls are selectable in notebook 04.


In [ ]:
show_image("ris_two_leaves.png", "Synthetic control geometry; the failed actual-plant diagnostics below use different meshes.", width=800)


In [ ]:
rt = dataframe("ris_controls_23_8ghz/rt_comparison.csv")
ideal = rt.loc[~rt["diagnostic"]].copy()
diagnostics = rt.loc[rt["diagnostic"]].copy()
assert len(ideal) == 14 and ideal["passed"].all()
assert len(diagnostics) == 12 and (~diagnostics["passed"]).all()
assert ideal[["solver_power_difference_db", "solver_phase_difference_deg"]].notna().all().all()
print(f"{len(ideal)} ideal polarized controls passed; {len(diagnostics)} diagnostic rows failed.")
print("Largest engine power difference (dB):", ideal["solver_power_difference_db"].abs().max())
print("Largest aligned phase difference (degrees):", ideal["solver_phase_difference_deg"].abs().max())
display(ideal[["case", "polarization", "sionna_paths", "differt_paths",
               "solver_power_difference_db", "solver_phase_difference_deg"]])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
labels = ideal["case"] + " / " + ideal["polarization"]
axes[0].bar(np.arange(len(ideal)), ideal["solver_power_difference_db"] * 1000, color="#137c8b")
axes[0].set(ylabel="DiffeRT − Sionna power (millidecibels)", title="Ideal controls: signed power gap")
axes[1].bar(np.arange(len(ideal)), ideal["solver_phase_difference_deg"], color="#d18738")
axes[1].set(ylabel="Aligned phase difference (degrees)", title="Phase convention already reconciled")
for axis in axes:
    axis.set_xticks(np.arange(len(ideal)), labels, rotation=70, ha="right", fontsize=8)
    axis.axhline(0, color="black", lw=0.6)
plt.show()


## Keep the failed diagnostics
These grazing/curved-leaf cases expose missing-path and numerical-tolerance issues.
A blank comparison means that the required coefficients were unavailable. It is not
evidence of agreement or zero attenuation. Process-local tolerance experiments from
the original study are outside this native-default table.


In [ ]:
display(diagnostics[["case", "polarization", "sionna_paths", "differt_paths",
                     "solver_power_difference_db", "passed"]])
assert diagnostics["solver_power_difference_db"].isna().all()
receipt = read_json("ris_controls_23_8ghz/verification.json")
print("Geometry gate:", receipt["geometry_gate"])
print("Production acceptance:", receipt["production_acceptance"])
print(receipt["reason"])


## A separate 60 GHz vegetation comparison
Select a case and polarization below. The historical focused study explicitly
replaced missed direct contributions with a checked direct-path calculation.
The plotted **corrected** totals must not be described as unmodified native totals.
Use the raw/corrected table to inspect that adjustment. R = reflection, T = transmission,
D1/D2 = configured maximum diffraction order. The two engines can discover different
edge families, so their difference is not solely a coefficient difference.


### The 60 GHz models
The open leaf screen is **80 × 120 mm**, centered at x = −150 mm; its radio
slab thickness is 0.2 mm. The square branch is **20 × 20 × 200 mm**.
TX/RX are at x = −1/+1 m and lie outside this detail crop. These are idealized
controls with nominal material properties, not realistic tree models.


In [ ]:
show_image("focused_leaf_branch.png", "Preserved leaf_branch mesh, rendered in Blender.", width=800)
show_image("focused_paths.png", "Historical path projections: the right panel uses a separate short link.")
display(compare_renders(ASSETS / "focused_wood_slab.png", ASSETS / "focused_wood_box.png",
    ("Single screen: 20 mm assigned radio slab thickness", "Closed 10 mm square branch: 5 mm assigned per crossing")))


In [ ]:
CASE, POLARIZATION = "leaf", "V"  # try branch or leaf_branch; V or H
focused = dataframe("focused_60ghz/results_table.csv")
selection = focused.loc[(focused.case == CASE) & (focused.polarization == POLARIZATION)]
assert not selection.empty, "Choose a case/polarization present in the table"
display(selection[["solver", "mode", "seed", "raw_paths", "accepted_paths",
                   "raw_excess_loss_db", "corrected_excess_loss_db", "direct_reference_power"]])
averages = selection.groupby(["mode", "solver"])["corrected_excess_loss_db"].mean().unstack()
averages.plot.bar(color=["#137c8b", "#d18738"], rot=0)
plt.ylabel("Corrected incoherent excess loss (dB)")
plt.title(f"{CASE}, {POLARIZATION}: historical focused study, 60 GHz")
plt.tight_layout()
plt.show()


## Optional fresh native diffraction fixtures
This independent section creates **new 3.5 GHz wedge fixtures**, not a rerun of the
23.8/60 GHz campaigns above. Inspect the fixture inventory first. The existing runner
preserves native baselines and records process failures and candidate-budget limits.

To run: prepare the pinned DiffeRT environment from [installation](../docs/install.md),
set its absolute interpreter path below, optionally set a separate Sionna interpreter,
then set `RUN_NATIVE = True`. The output is a fresh folder under `runs/`.
The default notebook execution performs analysis only.


In [ ]:
sys.path.insert(0, str(ROOT / "scripts"))
from diffraction_cases import pilot_cases
display(pd.DataFrame([{"fixture": c["name"], "frequency_hz": c["frequency_hz"],
                       "depth": c["solver"]["max_depth"], "expected": c["expected"]}
                      for c in pilot_cases()]))


In [ ]:
RUN_NATIVE = False
DIFFERT_PYTHON = None  # absolute path to a separately configured DiffeRT interpreter
SIONNA_PYTHON = None   # optional, separate Sionna interpreter
LLVM_LIBRARY = None   # optional Windows CPU LLVM library path

if RUN_NATIVE:
    import os, subprocess, tempfile
    if not DIFFERT_PYTHON or not Path(DIFFERT_PYTHON).is_file():
        raise ValueError("Configure DIFFERT_PYTHON before enabling a native run")
    (ROOT / "runs").mkdir(exist_ok=True)
    run_dir = Path(tempfile.mkdtemp(prefix="notebook-native-", dir=ROOT / "runs"))
    command = [sys.executable, str(ROOT / "scripts/run_diffraction.py"),
               "--differt-python", str(DIFFERT_PYTHON), "--output", str(run_dir)]
    if SIONNA_PYTHON:
        if Path(SIONNA_PYTHON).resolve() == Path(DIFFERT_PYTHON).resolve():
            raise ValueError("Use separate engine environments")
        command += ["--sionna-python", str(SIONNA_PYTHON)]
    if LLVM_LIBRARY:
        command += ["--llvm-library", str(LLVM_LIBRARY)]
    subprocess.run(command, cwd=ROOT, check=True,
                   creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0)
    import json
    native_receipt = json.loads((run_dir / "receipt.json").read_text())
    display(native_receipt)
else:
    print("Saved-result replay complete. Optional native fixtures are disabled.")
